# CMM synthetic validation

CMM uses flexmix (mixture of linear regressions), which assumes Gaussian conditional distributions.
Real TB data has binary mutation columns, handled via a Gaussian noise workaround.

This notebook tests whether CMM reliably recovers causal edges under two conditions:
- **binary → continuous** (mutation → MIC): the edge type we care about most
- **binary → binary** (mutation → mutation): the edge type dominating the real bootstrap results
```
Mixture structure: latent Z creates two subpopulations with different causal mechanisms.

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../external/cmm')

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from src_tb.data.synthetic import generate_synthetic, visualize_synthetic_dag, eval_recovery
from src_tb.cmm.cmm_utils import bootstrap_cmm, get_stable_edges, build_stable_bn
import pyagrum.lib.notebook as gnb

N_MUTATIONS = 4
N_SEEDS = 5
N_BOOTSTRAP = 15
THRESHOLD = 0.5

In [ ]:
# show the true DAG for the chosen n_mutations
_, features_synth, true_edges, binary_indices_synth = generate_synthetic(N_MUTATIONS)
true_bin_to_cont = {(s, t) for s, t in true_edges if t == 'Y'}
true_bin_to_bin  = {(s, t) for s, t in true_edges if t != 'Y'}

print(f"True edges: {true_edges}")
print(f"Binary to continuous: {true_bin_to_cont}")
print(f"Binary to binary:     {true_bin_to_bin}")
visualize_synthetic_dag(true_edges, features_synth)

In [ ]:
records = []
for seed in range(N_SEEDS):
    X, features_synth, true_edges, binary_indices_synth = generate_synthetic(N_MUTATIONS, seed=seed)
    cmm_list = bootstrap_cmm(X, set(), binary_indices_synth, n_runs=N_BOOTSTRAP)
    stable = get_stable_edges(cmm_list, features_synth, threshold=THRESHOLD)
    stable_set = set(zip(stable['source'], stable['target']))

    pr_bc, re_bc, f1_bc = eval_recovery(stable_set, true_bin_to_cont)
    pr_bb, re_bb, f1_bb = eval_recovery(stable_set, true_bin_to_bin)
    records.append({
        'seed': seed, 'stable_edges': stable_set,
        'bin->cont P': pr_bc, 'bin->cont R': re_bc, 'bin->cont F1': f1_bc,
        'bin->bin P':  pr_bb, 'bin->bin R':  re_bb, 'bin->bin F1':  f1_bb,
    })
    print(f"seed {seed} | {stable_set}")
    print(f"  bin->cont  P={pr_bc} R={re_bc} F1={f1_bc} | bin->bin  P={pr_bb} R={re_bb} F1={f1_bb}")

df = pd.DataFrame(records)

In [ ]:
cols = ['bin→cont precision', 'bin→cont recall', 'bin→cont F1',
        'bin→bin precision',  'bin→bin recall',  'bin→bin F1']
print("=== Mean ± std across 10 dataset seeds ===")
summary = df[cols].agg(['mean', 'std']).round(3)
print(summary.to_string())

In [ ]:
X_last, features_synth, true_edges, binary_indices_synth = generate_synthetic(N_MUTATIONS, seed=N_SEEDS - 1)
cmm_list_last = bootstrap_cmm(X_last, set(), binary_indices_synth, n_runs=N_BOOTSTRAP)
bn = build_stable_bn(cmm_list_last, features_synth, threshold=THRESHOLD)
gnb.showBN(bn, size="20")